In [ ]:
from os import listdir as ls
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import OUTPUTS_PATH, OXCGRT_LOCATION_CMAP

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

job_path = OUTPUTS_PATH / "48930936"
all_countries = [iso3 for iso3 in ls(job_path) if "oxcgrt" in ls(job_path / iso3)]
records = []
for iso3 in all_countries:
    analysis_path = job_path / iso3 / "oxcgrt"
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["mob_weights"].median(dim=("chain", "draw"))
    floor = float(idata.posterior["scale_floor"].median(dim=("chain", "draw")))
    best_policy = int(medians.argmax().item())
    total_weight = float(np.sum(medians))
    scale_factor = (1.0 - floor) / total_weight
    row = {
        "ISO_A3": iso3,
        "best_policy": best_policy,
    }
    for k, v in enumerate(medians):
        row[f"pol_{k}"] = float(v) * scale_factor
    records.append(row)
medians_df = pd.DataFrame.from_records(records)

world = world.merge(medians_df, on="ISO_A3", how="left")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
ax.set_xticks([])
ax.set_yticks([])
cmap = mcolors.ListedColormap(OXCGRT_LOCATION_CMAP.values(), name="policy_map")
world.plot(
    ax=ax, 
    column="best_policy", 
    edgecolor="black", 
    cmap=cmap,
)
missing = world[world["best_policy"].isna()]
missing.plot(
    ax=ax,
    facecolor="white",
    edgecolor="black",
    hatch="//",
    alpha=0.12,
)

In [ ]:
from emu_renewal.constants import OXCGRT_COLMAP, MOB_LOCATION_NAME_MAP

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(20, 18))
flat_axes = axes.ravel()
missing = world[world["best_policy"].isna()]
for a, ax in enumerate(flat_axes):
    ax.set_xticks([])
    ax.set_yticks([])
    world.plot(column=f"pol_{a}", cmap="Blues", ax=ax, edgecolor="black", linewidth=0.1)
    missing.plot(
        ax=ax,
        facecolor="white",
        edgecolor="black",
        hatch="//",
        alpha=0.12,
    )
    ax.set_title(MOB_LOCATION_NAME_MAP[OXCGRT_COLMAP["custom"][a]])
fig.tight_layout()